In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from datetime import datetime




# Helper to convert string date to datetime.date
sales_data = [
  ('A', datetime.strptime('2021-01-01', '%Y-%m-%d').date(), '1'),
  ('A', datetime.strptime('2021-01-01', '%Y-%m-%d').date(), '2'),
  ('A', datetime.strptime('2021-01-07', '%Y-%m-%d').date(), '2'),
  ('A', datetime.strptime('2021-01-10', '%Y-%m-%d').date(), '3'),
  ('A', datetime.strptime('2021-01-11', '%Y-%m-%d').date(), '3'),
  ('A', datetime.strptime('2021-01-11', '%Y-%m-%d').date(), '3'),
  ('B', datetime.strptime('2021-01-01', '%Y-%m-%d').date(), '2'),
  ('B', datetime.strptime('2021-01-02', '%Y-%m-%d').date(), '2'),
  ('B', datetime.strptime('2021-01-04', '%Y-%m-%d').date(), '1'),
  ('B', datetime.strptime('2021-01-11', '%Y-%m-%d').date(), '1'),
  ('B', datetime.strptime('2021-01-16', '%Y-%m-%d').date(), '3'),
  ('B', datetime.strptime('2021-02-01', '%Y-%m-%d').date(), '3'),
  ('C', datetime.strptime('2021-01-01', '%Y-%m-%d').date(), '3'),
  ('C', datetime.strptime('2021-01-01', '%Y-%m-%d').date(), '3'),
  ('C', datetime.strptime('2021-01-07', '%Y-%m-%d').date(), '3')
]

sales_schema = StructType([
    StructField("customer_id",StringType(),nullable=False),
    StructField("order_date",DateType(),nullable=False),
    StructField("product_id",StringType(),nullable=False)
])

sales_df = spark.createDataFrame(sales_data,schema=sales_schema)
 

menu_df = spark.createDataFrame(data=[
       ('1', 'sushi', '10'),
  ('2', 'curry', '15'),
  ('3', 'ramen', '12')
 ],schema=["product_id","product_name","price"])

# Convert join_date for members as well
members_data = [
    ('A', datetime.strptime('2021-01-07', '%Y-%m-%d').date()),
    ('B', datetime.strptime('2021-01-09', '%Y-%m-%d').date())
]
members_df = spark.createDataFrame(data=members_data, schema=["customer_id", "join_date"])




In [0]:
# sales_df.show() 

# menu_df.show() 


# 1. What is the total amount each customer spent at the restaurant?

sales_df.join(menu_df,"product_id","inner").groupBy("customer_id").agg(
    F.sum("price").alias("amount")
).show() 






In [0]:


#2.How many days has each customer visited the restaurant?


sales_df.groupBy("customer_id").agg(
    F.countDistinct("order_date").alias("number_of_days")
).show() 

In [0]:
from pyspark.sql import Window
#What was the first item from the menu purchased by each customer?


sales_df.withColumn("rank",F.dense_rank().over(Window.partitionBy("customer_id").orderBy("order_date"))).orderBy("order_date",ascending=True).filter(F.col("rank")==1).join(menu_df,"product_id","inner").groupBy("customer_id").agg(
    F.collect_set("product_name").alias("first_order")
).show()




In [0]:

# What is the most purchased item on the menu and how many times was it purchased by all customers?

sales_df.join(menu_df,"product_id","inner").groupBy("product_id").agg(
    F.sum("price").alias("totalPrice")
).orderBy("totalPrice",ascending=False).limit(1).show()

In [0]:
sales_df.join(menu_df,"product_id","inner").groupBy("customer_id","product_id").agg(
    F.count("product_id").alias("total")
).withColumn('rnk',F.dense_rank().over(Window.partitionBy("customer_id").orderBy(F.desc("total")))).filter(F.col("rnk")==1).join(menu_df,"product_id","inner").drop("rnk","product_id","price","total").show()